# 3. Normalization, integration weights and convergence

**Learning goals:** verify PDF normalization, compare grid and Monte Carlo integration, and preserve the measure when selecting events.

Run cells from top to bottom in a fresh Python kernel. Install the package and Jupyter
as explained in [the course guide](TUTORIALS.md). No external data files are needed.
Masses are in GeV, invariants in GeV², and daughter indices start at zero.
The small event counts and grid sizes keep this lesson practical on a CPU; they are
teaching settings, not a demonstrated precision choice for a physics analysis.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    DecayChannel, DecayModel, FitSession, NonResonant,
    Parameter, RealImag, Resonance, generate_toy,
)

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
x = Parameter.coefficient("NR.x", 0.55, owner="NR", bounds=(-2, 2), step=0.02)
y = Parameter.coefficient("NR.y", 0.30, owner="NR", bounds=(-2, 2), step=0.02)
components = [
    Resonance("rho", (0, 1), RealImag(1, 0), mass=0.7753, width=0.1491, spin=1),
    NonResonant(RealImag(x, y), name="NR"),
]
model = DecayModel(
    channel, components, normalization_method="square-dalitz",
    normalization_resolution=100, normalization_pair=(0, 1),
)
truth = {p.name: p.value for p in model.parameters}

## Integrate a normalized density

Every integral follows `mean(sample.weights * f)`, not a sum and not an unweighted mean.
The following check uses the same grid as the PDF: it verifies API use, but cannot establish
grid convergence. Independent grids below provide a stronger numerical comparison.
Component normalization and total PDF normalization are different operations: disabling
component rescaling never removes interference from the total normalization.

In [3]:
grid = model.normalization_sample
pdf = model.pdf()
integral = jnp.mean(grid.weights * pdf(grid.as_dict(), truth))
print("Integral on the construction grid:", float(integral))
np.testing.assert_allclose(integral, 1.0, rtol=1e-10)

## Increase the quadrature resolution

Compare raw coherent integrals with component rescaling disabled so changing grids does
not also redefine the coefficient convention. This is a small teaching scan: extend it
until changes are negligible for your analysis, and then repeat fits at converged settings.
Narrow resonances and sharp acceptance boundaries can need substantially finer grids.

In [4]:
resolutions = [60, 100, 160]
integrals = []
for resolution in resolutions:
    candidate = DecayModel(
        channel, components, normalize_components=False,
        normalization_method="square-dalitz", normalization_pair=(0, 1),
        normalization_resolution=resolution,
    )
    sample = candidate.normalization_sample
    value = float(jnp.mean(sample.weights * candidate.intensity(sample.as_dict(), truth)))
    integrals.append(value)
    print(f"resolution={resolution:3d}: raw integral={value:.8f}")
print("Relative change in last step:", abs(integrals[-1]/integrals[-2] - 1))

## Use a fixed external Monte Carlo sample

`PhaseSpaceMC` weights integrate `ds12 ds13 / (128*pi**3*M**2)`. The deterministic
grid uses `ds12 ds13`. Convert the weights when comparing absolute integrals.
The MC sample stays fixed throughout a fit. Repeat with independent seeds and larger
samples to assess numerical uncertainty, which Minuit does not propagate automatically.
Never substitute an unweighted signal toy for this proposal sample.

In [5]:
from dataclasses import replace
mc = model.generate_phase_space(50000, seed=31, include_momenta=False)
mc = replace(mc, weights=mc.weights * (128 * jnp.pi**3 * channel.parent_mass**2))
mc_model = DecayModel(channel, components, normalize_components=False,
                      normalization_sample=mc)
weighted_values = mc.weights * mc_model.intensity(mc.as_dict(), truth)
mc_integral = float(jnp.mean(weighted_values))
mc_error = float(jnp.std(weighted_values, ddof=1) / jnp.sqrt(mc.size))
print(f"MC raw integral: {mc_integral:.6f} +/- {mc_error:.6f} (MC standard error)")
print("Finest grid:", integrals[-1])

## Preserve integrals after a veto

Ordinary data selection preserves event weights. Integration selection additionally
multiplies retained weights by `N_retained/N_original` so the mean still represents the
original measure. The two calculations below must agree for the same sample.

In [6]:
from dalitzplotfitter import MassWindowVeto
veto = MassWindowVeto((0, 1), 0.45, 0.55)  # Mass bounds, in GeV.
mask = veto(mc.as_dict())
selected = veto.apply(mc, for_integration=True)
full_integral = jnp.mean(mc.weights * mask)
selected_integral = jnp.mean(selected.weights)
np.testing.assert_allclose(full_integral, selected_integral, rtol=1e-12)
print("Accepted area in ds12 ds13:", float(selected_integral))

## Try it yourself

1. Repeat the MC integral with three independent seeds.
2. Double the grid resolution and MC size separately.
3. Explain why selecting integration events without rescaling would overestimate the accepted area.

## Continue learning

[Next: acceptance and backgrounds](tutorial_04_acceptance_and_backgrounds.ipynb). Reference: [MC integration](../docs/mc_integration.md).

Return to [the course guide](TUTORIALS.md).